# Europe gas tracker - create data sets for download

In [4]:
import pygsheets # use 'pip install pygsheets', not maintained with conda

import pandas
import datetime
import numpy

# terminals

In [5]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')
spreadsheet = gc.open_by_key('1d0kLE0WmAn9b4XdugffiEaAHGWy6EhyF7zY1DM12zCc')
#spreadsheet = gc.open_by_key('1MrghwBeCz8Tzgua7CWGg_KoXKVZsV7r0kHMYHYqnNTg') # for July 2022 version update (terminals)

terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A2')
terms_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
terms_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()
terms_copyright_df = spreadsheet.worksheet('title', 'Copyright - EGT').get_as_df()

terms_region_dict_df = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

/Users/baird/mambaforge/envs/gem/lib/python3.10/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up terminals

In [6]:
# remove oil export terminals
terms_df_orig = terms_df_orig[terms_df_orig['Fuel']=='LNG']
# remove anything without a wiki page
terms_df_orig = terms_df_orig[terms_df_orig['Wiki']!='']

## subset columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [7]:
terms_dict_df_include = terms_dict_df.copy()[terms_dict_df.copy()['IncludeWithEGTDataRelease']=='Yes']
terms_dict_df_include = terms_dict_df_include.sort_values('EGTDataReleaseColumnOrder', ascending=True)
terms_dict_df_include = terms_dict_df_include[['VariableName','Definition']]

In [8]:
terms_df_subset = terms_df_orig.copy()[terms_dict_df_include['VariableName'].tolist()]
terms_df_subset.sort_values('ComboID', inplace=True)

## subset those in Europe Gas Tracker

In [9]:
terms_df_subset = terms_df_subset.loc[terms_df_subset.Country.isin(
    terms_region_dict_df.loc[terms_region_dict_df.EuroGasTracker=='Yes']['Country'].tolist())]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [10]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('GEM-Europe-Gas-Tracker-LNG-Terminals-March-2023-updated-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')

terms_df_subset.to_excel(excel_writer, sheet_name='LNG Terminals '+str(datetime.date.today()), index=False)
terms_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
terms_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
terms_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

excel_writer.close()

# pipelines

In [11]:
fuel_type = 'Gas'
#fuel_type = 'Oil'
#fuel_type = 'Oil-and-Gas'

gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')

#spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
spreadsheet = gc.open_by_key('1PKsCoVnfnCEalDBOF0Fmny0-pg1qy86DoReNHI-97WM') # Mar 2023 Euro gas report
#spreadsheet[1] "Gas Pipelines" tab is the second index
gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A2')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A2')
#owners = spreadsheet[3].get_as_df()

#remove liquid pipelines you don't want to distribute
remove_fuels = ['Carbon']
oil_pipes = oil_pipes[~oil_pipes['Fuel'].isin(remove_fuels)]

pipes_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
pipes_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()

if fuel_type == 'Gas':
    pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)
if fuel_type == 'Oil':
    pipes_df_orig = oil_pipes.copy()
if fuel_type == 'Oil-and-Gas':
    pipes_df_orig = pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

pipes_copyright_df = spreadsheet.worksheet('title', 'Copyright - EGT').get_as_df()

/Users/baird/mambaforge/envs/gem/lib/python3.10/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


In [12]:
pipes_region_dict_df = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

## clean up pipelines

In [13]:
# clean up rows that should not be distributed
pipes_df_orig = pipes_df_orig[pipes_df_orig['Status']!='N/A']
pipes_df_orig = pipes_df_orig[pipes_df_orig['PipelineName']!='']
pipes_df_orig = pipes_df_orig[pipes_df_orig['Wiki']!='']

In [14]:
status_in_dev = ['Proposed', 
                 'Construction', 
                 'Shelved', 'Operating', 
                 'Mothballed', 
                 'Cancelled', 
                 'Retired', 
                 'Idle']
no_route_options = [
    'Unavailable', 
    'Capacity expansion only', 
    'Bidirectionality upgrade only',
    'Short route (< 100 km)', 
    'N/A',
    ''
]

# filter for the statuses above in the status_in_dev list (modify as desired)
#gas_pipes = gas_pipes[gas_pipes['Status'].str.lower().isin(status_in_dev)]

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [15]:
pipes_dict_df_include = pipes_dict_df.loc[(pipes_dict_df['IncludeWithEGTDataRelease']=='Yes')&
                                             (pipes_dict_df['GasVariable']=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('EGTDataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

### subset countries in Euro gas tracker

In [16]:
pipes_df_subset = pipes_df_orig.loc[
    ~pipes_df_orig.Countries.apply(
        lambda x: set(x.split(', ')).isdisjoint(set(pipes_region_dict_df.loc[pipes_region_dict_df.EuroGasTracker=='Yes']['Country'].tolist()))
    )]

# deal with H2 pipelines

In [17]:
# copy anything with H2Status and Fuel==Gas into new row
pipes_df_subset_gas_to_h2 = pipes_df_subset.copy()[(pipes_df_subset.H2Status!='')&
                                                   (pipes_df_subset.Fuel=='Gas')]
pipes_df_subset_gas_to_h2['Status'] = pipes_df_subset_gas_to_h2['H2Status']
pipes_df_subset_gas_to_h2['StartYearEarliest'] = pipes_df_subset_gas_to_h2['H2StartYear']
pipes_df_subset_gas_to_h2['Fuel'] = 'Hydrogen'

In [18]:
pipes_df_subset_with_h2 = pandas.concat([pipes_df_subset,
                                         pipes_df_subset_gas_to_h2])

In [19]:
pipes_df_subset_with_h2 = pipes_df_subset_with_h2[pipes_dict_df_include['VariableName'].tolist()]
pipes_df_subset_with_h2.sort_values('ProjectID', inplace=True)

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [20]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('GEM-Europe-Gas-Tracker-'+fuel_type+'-and-Hydrogen-Pipelines-March-2023-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')

pipes_df_subset_with_h2.to_excel(excel_writer, sheet_name='Pipelines '+str(datetime.date.today()), index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
if fuel_type in ['Oil','Gas']:
    pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})gi

excel_writer.close()